
# Amazon Laptop Product Database — Preprocessing v2

This notebook freezes the data-preprocessing direction for the Industry Project.

## Objective

Build a **small, high-confidence, structured laptop product database** that is good enough for deterministic product retrieval and grounded response generation.

We deliberately prefer **precision, simplicity, and project completeness** over maximum Amazon catalogue recall.

Pipeline:

**Cleaned Electronics JSONL → high-confidence laptop category → valid price → obvious junk/accessory exclusion → canonical field extraction/normalisation → grounded product description construction → quality/completeness analysis → retrieval-ready product database**

## Frozen product schema

### Retrieval / structured fields
- `product_id`
- `title`
- `brand`
- `price`
- `ram_gb`
- `storage_gb`
- `screen_in`
- `gpu`
- `weight_kg`

### Additional grounded context
- `cpu`
- `os`
- `description`
- `average_rating`
- `rating_number`

### Audit fields
- `category_path`
- `quality_score`
- `strict_complete`

`usage` is intentionally **not** stored as an Amazon product attribute. It belongs to the conversational preference state and can be used by the final response layer together with `preference_notes`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install pyarrow

In [3]:

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 40)


## 1. Paths and configuration

In [4]:

PROJECT_DIR = Path("/content/drive/MyDrive/UTS Courses/Industrial Project")
INPUT_JSONL = PROJECT_DIR / "meta_Electronics_cleaned.jsonl"

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ALL_OUTPUT = PROCESSED_DIR / "laptop_catalogue_v2_all.parquet"
DATABASE_OUTPUT = PROCESSED_DIR / "laptop_product_database_v2.parquet"
DATABASE_CSV = PROCESSED_DIR / "laptop_product_database_v2.csv"
STRICT_OUTPUT = PROCESSED_DIR / "laptop_product_database_v2_strict_complete.parquet"
COVERAGE_OUTPUT = PROCESSED_DIR / "laptop_schema_coverage_v2.csv"

assert INPUT_JSONL.exists(), f"Input file not found: {INPUT_JSONL}"
print("Input:", INPUT_JSONL)
print("Processed directory:", PROCESSED_DIR)


Input: /content/drive/MyDrive/UTS Courses/Industrial Project/meta_Electronics_cleaned.jsonl
Processed directory: /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed



## 2. High-confidence laptop filter

We keep only records that:

1. have a positive price;
2. have the exact `Laptops` category in the Amazon category hierarchy;
3. do not contain an obvious accessory/software title signal.

This intentionally sacrifices recall.


In [5]:

NEGATIVE_CATEGORY_TERMS = {
    "laptop accessories",
    "laptop replacement parts",
    "skins & decals",
    "bags, cases & sleeves",
    "laptop network adapters",
}

NEGATIVE_TITLE_PATTERNS = [
    re.compile(r"(?i)\bwireless\s+card\b"),
    re.compile(r"(?i)\breplacement\b"),
    re.compile(r"(?i)\b(?:hard\s+shell\s+)?case\b"),
    re.compile(r"(?i)\bcover\b"),
    re.compile(r"(?i)\bskin\b"),
    re.compile(r"(?i)\bsleeve\b"),
    re.compile(r"(?i)\bbag\b"),
    re.compile(r"(?i)\badapter\b"),
    re.compile(r"(?i)\bcharger\b"),
    re.compile(r"(?i)\bbattery\b"),
    re.compile(r"(?i)\bdocking?\s+station\b"),
    re.compile(r"(?i)\bscreen\s+protector\b"),
    re.compile(r"(?i)\bdecal\b"),
    re.compile(r"(?i)\bsticker\b"),
    re.compile(r"(?i)\bpublisher\s+pro\b"),
]

def normalise_categories(categories):
    if not isinstance(categories, list):
        return []
    return [str(x).strip() for x in categories if x is not None]

def has_obvious_negative_title(title):
    text = str(title or "")
    return any(p.search(text) for p in NEGATIVE_TITLE_PATTERNS)


## 3. Canonical brand normalisation

In [6]:

CANONICAL_BRANDS = [
    "Acer", "Apple", "ASUS", "Dell", "Gateway", "Gigabyte", "HP",
    "Huawei", "Lenovo", "LG", "Microsoft", "MSI", "Panasonic",
    "Razer", "Samsung", "Toshiba", "Other",
]

BRAND_PATTERNS = [
    ("Apple", re.compile(r"(?i)\b(?:apple|macbook)\b")),
    ("Acer", re.compile(r"(?i)\b(?:acer|aspire|swift|nitro|predator)\b")),
    ("ASUS", re.compile(r"(?i)\b(?:asus|asustek|vivobook|zenbook|rog|zephyrus|tuf)\b")),
    ("Dell", re.compile(r"(?i)\b(?:dell|alienware|latitude|inspiron|xps)\b")),
    ("Gateway", re.compile(r"(?i)\bgateway\b")),
    ("Gigabyte", re.compile(r"(?i)\bgigabyte\b")),
    ("HP", re.compile(r"(?i)\b(?:hp|hewlett[\s-]*packard|elitebook|probook|pavilion|envy|victus|omen)\b")),
    ("Huawei", re.compile(r"(?i)\b(?:huawei|matebook)\b")),
    ("Lenovo", re.compile(r"(?i)\b(?:lenovo|thinkpad|ideapad|thinkbook|legion|yoga)\b")),
    ("LG", re.compile(r"(?i)\b(?:lg|gram)\b")),
    ("Microsoft", re.compile(r"(?i)\b(?:microsoft|surface)\b")),
    ("MSI", re.compile(r"(?i)\bmsi\b")),
    ("Panasonic", re.compile(r"(?i)\b(?:panasonic|toughbook)\b")),
    ("Razer", re.compile(r"(?i)\brazer\b")),
    ("Samsung", re.compile(r"(?i)\bsamsung\b")),
    ("Toshiba", re.compile(r"(?i)\b(?:toshiba|satellite|portege|tecra)\b")),
]

def clean_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " ".join(clean_text(v) for v in value)
    if isinstance(value, dict):
        return " ".join(f"{k}: {clean_text(v)}" for k, v in value.items())
    return str(value)

def normalise_details(details):
    if not isinstance(details, dict):
        return {}
    return {str(k).strip().lower(): v for k, v in details.items()}

def exact_detail_value(details, candidate_keys):
    d = normalise_details(details)
    for key in candidate_keys:
        if key in d and d[key] not in (None, "", [], {}):
            return clean_text(d[key]).strip()
    return None

def canonical_brand_from_text(text):
    s = str(text or "")
    for brand, pattern in BRAND_PATTERNS:
        if pattern.search(s):
            return brand
    return None

def extract_brand(record):
    value = canonical_brand_from_text(record.get("title"))
    if value:
        return value

    brand_detail = exact_detail_value(record.get("details", {}), ["brand", "brand name"])
    value = canonical_brand_from_text(brand_detail)
    if value:
        return value

    value = canonical_brand_from_text(record.get("store"))
    if value:
        return value

    return "Other"


## 4. RAM, storage and screen extraction

In [7]:

def gb_from_value(number, unit):
    x = float(number)
    if unit.upper() == "TB":
        x *= 1024
    return int(round(x))

ALLOWED_RAM_GB = {2, 4, 6, 8, 12, 16, 20, 24, 32, 36, 40, 48, 64, 96, 128}

RAM_PATTERNS = [
    re.compile(r"(?i)\b(\d{1,3})\s*(GB|TB)\s*(?:RAM|DDR[345]?|LPDDR[345X]?)\b"),
    re.compile(r"(?i)\bRAM(?:\s+Memory)?\s*[:\-]?\s*(\d{1,3})\s*(GB|TB)\b"),
]

def parse_ram_from_text(text):
    s = clean_text(text)
    for pat in RAM_PATTERNS:
        m = pat.search(s)
        if m:
            value = gb_from_value(m.group(1), m.group(2))
            if value in ALLOWED_RAM_GB:
                return value
    return None

def extract_ram_gb(record):
    value = parse_ram_from_text(record.get("title"))
    if value is not None:
        return value

    detail_value = exact_detail_value(
        record.get("details", {}),
        ["ram memory installed size", "ram memory size", "installed ram", "installed memory"]
    )
    value = parse_ram_from_text(detail_value)
    if value is not None:
        return value

    return parse_ram_from_text(record.get("features"))

STORAGE_PATTERNS = [
    re.compile(r"(?i)\b(\d{2,5}(?:\.\d+)?)\s*(GB|TB)\s*(?:SSD|HDD|eMMC|NVMe|storage|solid state|hard drive)\b"),
    re.compile(r"(?i)\b(?:SSD|HDD|eMMC|NVMe|storage|solid state|hard drive)\s*[:\-]?\s*(\d{2,5}(?:\.\d+)?)\s*(GB|TB)\b"),
]

def parse_storage_from_text(text):
    s = clean_text(text)
    found = []
    for pat in STORAGE_PATTERNS:
        for m in pat.finditer(s):
            value = gb_from_value(m.group(1), m.group(2))
            if 32 <= value <= 8192:
                found.append(value)
    if not found:
        return None
    unique_values = []
    for value in found:
        if value not in unique_values:
            unique_values.append(value)
    total = sum(unique_values)
    return total if 32 <= total <= 8192 else None

def extract_storage_gb(record):
    value = parse_storage_from_text(record.get("title"))
    if value is not None:
        return value

    detail_value = exact_detail_value(
        record.get("details", {}),
        ["hard disk size", "hard drive", "memory storage capacity", "storage capacity", "solid state drive capacity"]
    )
    value = parse_storage_from_text(detail_value)
    if value is not None:
        return value

    return parse_storage_from_text(record.get("features"))

SCREEN_PATTERNS = [
    re.compile(r'(?i)\b(\d{2}(?:\.\d{1,2})?)\s*(?:-| )?(?:inch|inches|")'),
    re.compile(r"(?i)\b(\d{2}(?:\.\d{1,2})?)\s*[- ]?in\b"),
]

def parse_screen_from_text(text):
    s = clean_text(text)
    for pat in SCREEN_PATTERNS:
        for m in pat.finditer(s):
            value = float(m.group(1))
            if 10.0 <= value <= 20.0:
                return round(value, 2)
    return None

def extract_screen_in(record):
    title_value = parse_screen_from_text(record.get("title"))
    detail_value_raw = exact_detail_value(record.get("details", {}), ["screen size", "display size"])
    detail_value = parse_screen_from_text(detail_value_raw)

    if title_value is not None and detail_value is not None:
        if abs(title_value - detail_value) > 0.75:
            return None
        return title_value
    if title_value is not None:
        return title_value
    if detail_value is not None:
        return detail_value
    return parse_screen_from_text(record.get("features"))


## 5. GPU, CPU and OS normalisation

In [8]:

GPU_MODEL_PATTERNS = [
    re.compile(r"(?i)\bRTX\s*(2050|2060|2070|2080|3050|3060|3070|3080|3090|4050|4060|4070|4080|4090)\s*(Ti)?\b"),
    re.compile(r"(?i)\bGTX\s*(1050|1060|1070|1080|1650|1660)\s*(Ti)?\b"),
]

def canonicalise_gpu_text(text):
    s = clean_text(text)
    if not s:
        return None
    for pat in GPU_MODEL_PATTERNS:
        m = pat.search(s)
        if m:
            prefix = "RTX" if "RTX" in m.group(0).upper() else "GTX"
            ti = " Ti" if m.group(2) else ""
            return f"{prefix} {m.group(1)}{ti}"
    if re.search(r"(?i)\bIntel\s+Iris\s+Xe\b", s):
        return "Intel Iris Xe"
    if re.search(r"(?i)\bIntel\s+Arc\b", s):
        return "Intel Arc"
    if re.search(r"(?i)\b(?:AMD\s+)?Radeon\b", s):
        return "AMD Radeon"
    if re.search(r"(?i)\b(?:integrated|intel\s+uhd|intel\s+hd\s+graphics|iris\s+plus|integrated\s+graphics)\b", s):
        return "Integrated"
    return "Other"

def extract_gpu(record):
    title_gpu = canonicalise_gpu_text(record.get("title"))
    if title_gpu and title_gpu != "Other":
        return title_gpu

    detail_value = exact_detail_value(
        record.get("details", {}),
        ["graphics coprocessor", "graphics card description", "graphics card", "gpu model"]
    )
    detail_gpu = canonicalise_gpu_text(detail_value)
    if detail_gpu and detail_gpu != "Other":
        return detail_gpu

    feature_gpu = canonicalise_gpu_text(record.get("features"))
    if feature_gpu and feature_gpu != "Other":
        return feature_gpu

    if title_gpu == "Other" or detail_gpu == "Other" or feature_gpu == "Other":
        return "Other"
    return None

CPU_TITLE_PATTERNS = [
    re.compile(r"(?i)\bIntel\s+Core\s+i[3579][\-\s]?[A-Za-z0-9]+"),
    re.compile(r"(?i)\bIntel\s+Core\s+Ultra\s+[3579](?:\s+[A-Za-z0-9-]+)?"),
    re.compile(r"(?i)\bIntel\s+(?:Celeron|Pentium)\s+[A-Za-z0-9-]+"),
    re.compile(r"(?i)\bAMD\s+Ryzen\s+[3579]\s+[A-Za-z0-9-]+"),
    re.compile(r"(?i)\bApple\s+M[1234](?:\s+(?:Pro|Max|Ultra))?"),
]

def clean_cpu_value(value):
    text = clean_text(value).strip()
    if not text:
        return None
    if re.fullmatch(r"(?i)\d+(?:\.\d+)?\s*GHz", text):
        return None
    return text[:100]

def extract_cpu(record):
    detail_value = exact_detail_value(record.get("details", {}), ["cpu model", "processor model", "processor"])
    value = clean_cpu_value(detail_value)
    if value:
        return value
    title = clean_text(record.get("title"))
    for pat in CPU_TITLE_PATTERNS:
        m = pat.search(title)
        if m:
            return m.group(0).strip()[:100]
    return None

def canonicalise_os(text):
    s = clean_text(text)
    if not s:
        return None
    if re.search(r"(?i)\bWindows\s*11\b", s):
        return "Windows 11"
    if re.search(r"(?i)\bWindows\s*10\b", s):
        return "Windows 10"
    if re.search(r"(?i)\bChrome\s?OS\b", s):
        return "Chrome OS"
    if re.search(r"(?i)\b(?:macOS|Mac\s+OS|OS\s+X)\b", s):
        return "macOS"
    if re.search(r"(?i)\bLinux\b", s):
        return "Linux"
    return "Other"

def extract_os(record):
    detail_value = exact_detail_value(record.get("details", {}), ["operating system"])
    value = canonicalise_os(detail_value)
    if value and value != "Other":
        return value
    title_value = canonicalise_os(record.get("title"))
    if title_value and title_value != "Other":
        return title_value
    feature_value = canonicalise_os(record.get("features"))
    if feature_value and feature_value != "Other":
        return feature_value
    if value == "Other" or title_value == "Other" or feature_value == "Other":
        return "Other"
    return None


## 6. Weight extraction

In [9]:

WEIGHT_PATTERN = re.compile(r"(?i)\b(\d+(?:\.\d+)?)\s*(kg|kilograms?|g|grams?|lb|lbs|pounds?|oz|ounces?)\b")

def weight_to_kg(number, unit):
    value = float(number)
    u = unit.lower()
    if u in {"kg", "kilogram", "kilograms"}:
        kg = value
    elif u in {"g", "gram", "grams"}:
        kg = value / 1000.0
    elif u in {"lb", "lbs", "pound", "pounds"}:
        kg = value * 0.45359237
    elif u in {"oz", "ounce", "ounces"}:
        kg = value * 0.028349523125
    else:
        return None
    return round(kg, 2) if 0.5 <= kg <= 6.0 else None

def parse_weight(text):
    s = clean_text(text)
    if not s:
        return None
    m = WEIGHT_PATTERN.search(s)
    return weight_to_kg(m.group(1), m.group(2)) if m else None

def extract_weight_kg(record):
    detail_value = exact_detail_value(record.get("details", {}), ["item weight"])
    value = parse_weight(detail_value)
    if value is not None:
        return value
    return parse_weight(record.get("title"))


## 7. Grounded product description construction

In [10]:

DESCRIPTION_DETAIL_KEYS = [
    "model name", "item model number", "brand", "screen size", "cpu model",
    "processor", "ram memory installed size", "hard disk size",
    "graphics coprocessor", "operating system", "item weight",
]

def normalise_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()

def build_product_description(record, max_chars=3000):
    parts = []
    title = normalise_space(record.get("title"))
    if title:
        parts.append(f"Title: {title}")

    features = record.get("features")
    if isinstance(features, list):
        clean_features = [normalise_space(x) for x in features if normalise_space(x)][:8]
        if clean_features:
            parts.append("Features: " + " | ".join(clean_features))

    description = record.get("description")
    if isinstance(description, list):
        clean_description = [normalise_space(x) for x in description if normalise_space(x)][:4]
        if clean_description:
            parts.append("Description: " + " | ".join(clean_description))

    details = normalise_details(record.get("details", {}))
    selected_details = []
    for wanted in DESCRIPTION_DETAIL_KEYS:
        if wanted in details and details[wanted] not in (None, "", [], {}):
            selected_details.append(f"{wanted}: {normalise_space(clean_text(details[wanted]))}")
    if selected_details:
        parts.append("Product details: " + " | ".join(selected_details))

    combined = normalise_space(" ".join(parts))
    return combined[:max_chars] if combined else None


## 8. Extraction smoke tests

In [11]:

TEST_RECORDS = [
    {
        "title": 'Acer Aspire 3 A314 Slim Laptop 14" AMD Ryzen 5 7520U 8GB LPDDR5 512GB NVMe SSD AMD Radeon Graphics Windows 11 Home',
        "store": "Acer", "features": [], "description": [],
        "details": {"Item Weight": "3.09 pounds"}
    },
    {
        "title": 'Lenovo Legion 5 Gaming Laptop 15.6" AMD Ryzen 7 5800H 16GB RAM 512GB Storage NVIDIA GeForce RTX 3050Ti Windows 10',
        "store": "Lenovo", "features": [], "description": [],
        "details": {"Item Weight": "5.29 lbs"}
    },
    {
        "title": 'Apple MacBook Air 13-inch 8GB RAM 256GB SSD',
        "store": "Apple", "features": [], "description": [],
        "details": {"Operating System": "Mac OS", "Item Weight": "2.8 pounds"}
    },
]

for i, rec in enumerate(TEST_RECORDS, start=1):
    print("=" * 100)
    print("TEST", i)
    print({
        "brand": extract_brand(rec),
        "ram_gb": extract_ram_gb(rec),
        "storage_gb": extract_storage_gb(rec),
        "screen_in": extract_screen_in(rec),
        "gpu": extract_gpu(rec),
        "weight_kg": extract_weight_kg(rec),
        "cpu": extract_cpu(rec),
        "os": extract_os(rec),
        "description": build_product_description(rec)[:180],
    })
print("Smoke tests completed.")


TEST 1
{'brand': 'Acer', 'ram_gb': 8, 'storage_gb': 512, 'screen_in': 14.0, 'gpu': 'AMD Radeon', 'weight_kg': 1.4, 'cpu': 'AMD Ryzen 5 7520U', 'os': 'Windows 11', 'description': 'Title: Acer Aspire 3 A314 Slim Laptop 14" AMD Ryzen 5 7520U 8GB LPDDR5 512GB NVMe SSD AMD Radeon Graphics Windows 11 Home Product details: item weight: 3.09 pounds'}
TEST 2
{'brand': 'Lenovo', 'ram_gb': 16, 'storage_gb': 512, 'screen_in': 15.6, 'gpu': 'RTX 3050 Ti', 'weight_kg': 2.4, 'cpu': 'AMD Ryzen 7 5800H', 'os': 'Windows 10', 'description': 'Title: Lenovo Legion 5 Gaming Laptop 15.6" AMD Ryzen 7 5800H 16GB RAM 512GB Storage NVIDIA GeForce RTX 3050Ti Windows 10 Product details: item weight: 5.29 lbs'}
TEST 3
{'brand': 'Apple', 'ram_gb': 8, 'storage_gb': 256, 'screen_in': 13.0, 'gpu': 'Other', 'weight_kg': 1.27, 'cpu': None, 'os': 'macOS', 'description': 'Title: Apple MacBook Air 13-inch 8GB RAM 256GB SSD Product details: operating system: Mac OS | item weight: 2.8 pounds'}
Smoke tests completed.


## 9. Stream the full dataset

In [12]:

rows = []
scanned = 0
category_price_candidates = 0
excluded_by_title = 0
kept = 0
bad_json = 0
extraction_errors = 0
error_examples = []

def safe_extract(fn, record, field):
    global extraction_errors
    try:
        return fn(record)
    except Exception as e:
        extraction_errors += 1
        if len(error_examples) < 20:
            error_examples.append({
                "field": field,
                "product_id": record.get("parent_asin"),
                "title": str(record.get("title") or "")[:200],
                "error_type": type(e).__name__,
                "error": str(e)[:400],
            })
        return None

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        scanned += 1
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            bad_json += 1
            continue

        categories = normalise_categories(rec.get("categories"))
        cats_lower = {c.lower() for c in categories}
        price = rec.get("price")
        valid_price = isinstance(price, (int, float)) and not isinstance(price, bool) and price > 0

        base_candidate = (
            "laptops" in cats_lower
            and valid_price
            and not any(any(term in c.lower() for term in NEGATIVE_CATEGORY_TERMS) for c in categories)
        )
        if base_candidate:
            category_price_candidates += 1
        if not base_candidate:
            continue
        if has_obvious_negative_title(rec.get("title")):
            excluded_by_title += 1
            continue

        kept += 1
        rows.append({
            "product_id": rec.get("parent_asin"),
            "title": rec.get("title"),
            "brand": safe_extract(extract_brand, rec, "brand"),
            "price": float(rec["price"]),
            "ram_gb": safe_extract(extract_ram_gb, rec, "ram_gb"),
            "storage_gb": safe_extract(extract_storage_gb, rec, "storage_gb"),
            "screen_in": safe_extract(extract_screen_in, rec, "screen_in"),
            "gpu": safe_extract(extract_gpu, rec, "gpu"),
            "weight_kg": safe_extract(extract_weight_kg, rec, "weight_kg"),
            "cpu": safe_extract(extract_cpu, rec, "cpu"),
            "os": safe_extract(extract_os, rec, "os"),
            "description": safe_extract(build_product_description, rec, "description"),
            "average_rating": rec.get("average_rating"),
            "rating_number": rec.get("rating_number"),
            "category_path": " > ".join(categories),
        })

        if scanned % 250_000 == 0:
            print(f"Scanned {scanned:,} | base candidates {category_price_candidates:,} | title exclusions {excluded_by_title:,} | kept {kept:,}")

print("\nFinished.")
print("Rows scanned:", f"{scanned:,}")
print("Bad JSON rows:", f"{bad_json:,}")
print("Laptop category + price candidates:", f"{category_price_candidates:,}")
print("Excluded by obvious negative title:", f"{excluded_by_title:,}")
print("Rows kept:", f"{kept:,}")
print("Extraction errors:", f"{extraction_errors:,}")
if error_examples:
    display(pd.DataFrame(error_examples))
else:
    print("No extraction errors.")



Finished.
Rows scanned: 1,610,012
Bad JSON rows: 0
Laptop category + price candidates: 10,308
Excluded by obvious negative title: 522
Rows kept: 9,786
Extraction errors: 0
No extraction errors.


## 10. DataFrame cleanup and sanity checks

In [13]:

catalogue = pd.DataFrame(rows)
catalogue = catalogue.dropna(subset=["product_id", "title", "price"]).copy()
catalogue = catalogue[catalogue["price"] > 0].copy()
catalogue = (
    catalogue
    .sort_values(["product_id", "rating_number"], ascending=[True, False])
    .drop_duplicates("product_id", keep="first")
    .reset_index(drop=True)
)
print("Rows after cleanup:", f"{len(catalogue):,}")
print("Unique product IDs:", f"{catalogue['product_id'].nunique():,}")
display(catalogue.head(30))


## 11. Sanity checks for contamination

In [14]:

print("Brand distribution:")
display(catalogue["brand"].value_counts(dropna=False).rename_axis("brand").reset_index(name="count"))

print("\nRAM values:")
display(catalogue["ram_gb"].value_counts(dropna=False).sort_index().rename_axis("ram_gb").reset_index(name="count").head(50))

print("\nStorage summary:")
display(catalogue["storage_gb"].describe())

print("\nScreen summary:")
display(catalogue["screen_in"].describe())

print("\nWeight summary:")
display(catalogue["weight_kg"].describe())

print("\nGPU distribution:")
display(catalogue["gpu"].value_counts(dropna=False).head(30).rename_axis("gpu").reset_index(name="count"))


Brand distribution:


,brand,count
0,HP,2818
1,Lenovo,2219
2,Dell,1952
3,ASUS,920
4,Acer,542
5,Apple,338
6,Other,264
7,MSI,251
8,Samsung,127
9,Microsoft,124



RAM values:


,ram_gb,count
0,2.0,99
1,4.0,1536
2,6.0,47
3,8.0,2647
4,12.0,456
5,16.0,2314
6,20.0,121
7,24.0,45
8,32.0,771
9,36.0,7



Storage summary:


...

,screen_in
count,9547.000000
mean,14.671849
std,1.515393
min,10.000000
25%,14.000000
50%,15.600000
75%,15.600000
max,18.400000



Weight summary:


,weight_kg
count,8943.000000
mean,1.960793
std,0.668935
min,0.500000
25%,1.500000
50%,1.810000
75%,2.290000
max,5.990000



GPU distribution:


,gpu,count
0,Integrated,5052
1,Other,1304
2,AMD Radeon,1214
3,Intel Iris Xe,1019
4,RTX 3060,181
5,GTX 1650,172
6,RTX 3050 Ti,124
7,RTX 3050,96
8,RTX 3070,86
9,GTX 1660 Ti,67


## 12. Schema coverage

In [15]:

SCHEMA_FIELDS = ["brand", "price", "ram_gb", "storage_gb", "screen_in", "gpu", "weight_kg", "cpu", "os", "description"]
coverage_rows = []
for col in SCHEMA_FIELDS:
    if col == "brand":
        usable = catalogue[col].notna() & catalogue[col].ne("Other")
    else:
        usable = catalogue[col].notna()
    coverage_rows.append({
        "field": col,
        "non_null_count": int(catalogue[col].notna().sum()),
        "usable_count": int(usable.sum()),
        "non_null_pct": round(100 * catalogue[col].notna().mean(), 2),
        "usable_pct": round(100 * usable.mean(), 2),
    })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)


,field,non_null_count,usable_count,non_null_pct,usable_pct
0,brand,9786,9522,100.00,97.30
1,price,9786,9786,100.00,100.00
2,ram_gb,8265,8265,84.46,84.46
3,storage_gb,5828,5828,59.55,59.55
4,screen_in,9547,9547,97.56,97.56
5,gpu,9786,9786,100.00,100.00
6,weight_kg,8943,8943,91.39,91.39
7,cpu,9267,9267,94.70,94.70
8,os,9786,9786,100.00,100.00
9,description,9786,9786,100.00,100.00



## 13. Completeness / retrieval-ready policy

We export two useful subsets.

### Retrieval-ready database
Required: price, RAM, storage, screen, GPU, and description.
Brand may be `Other`; weight/CPU/OS may be missing.

### Strict-complete subset
Additionally requires weight, CPU, and OS.

If strict-complete is still around 3,000+ rows, excellent. If not, do **not** invent values just to hit a target.


In [16]:

RETRIEVAL_REQUIRED = ["price", "ram_gb", "storage_gb", "screen_in", "gpu", "description"]
STRICT_EXTRA = ["weight_kg", "cpu", "os"]

catalogue["quality_score"] = catalogue[SCHEMA_FIELDS].notna().sum(axis=1)
retrieval_ready_mask = catalogue[RETRIEVAL_REQUIRED].notna().all(axis=1)
strict_complete_mask = retrieval_ready_mask & catalogue[STRICT_EXTRA].notna().all(axis=1)

catalogue["strict_complete"] = strict_complete_mask
retrieval_ready = catalogue.loc[retrieval_ready_mask].copy()
strict_complete = catalogue.loc[strict_complete_mask].copy()

print("All cleaned high-confidence laptops:", f"{len(catalogue):,}")
print("Retrieval-ready database:", f"{len(retrieval_ready):,}")
print("Strict-complete subset:", f"{len(strict_complete):,}")

if len(strict_complete) >= 3000:
    print("PASS — strict-complete subset is >= 3,000 products.")
else:
    print("Below 3,000. Use the retrieval-ready database and keep weight/CPU/OS optional rather than fabricating values.")


All cleaned high-confidence laptops: 9,786
Retrieval-ready database: 4,938
Strict-complete subset: 4,331
PASS — strict-complete subset is >= 3,000 products.


## 14. Inspect the final MVP-style product table

In [17]:

DISPLAY_COLS = [
    "product_id", "title", "brand", "price", "ram_gb", "storage_gb",
    "screen_in", "gpu", "weight_kg", "cpu", "os", "average_rating", "rating_number"
]
display(
    retrieval_ready
    .sort_values(["quality_score", "rating_number"], ascending=[False, False])
    [DISPLAY_COLS]
    .head(100)
)


## 15. Export the product database

In [18]:

FINAL_COLUMNS = [
    "product_id", "title", "brand", "price", "ram_gb", "storage_gb", "screen_in",
    "gpu", "weight_kg", "cpu", "os", "description", "average_rating", "rating_number",
    "category_path", "quality_score", "strict_complete"
]

catalogue[FINAL_COLUMNS].to_parquet(ALL_OUTPUT, index=False)
retrieval_ready[FINAL_COLUMNS].to_parquet(DATABASE_OUTPUT, index=False)
retrieval_ready[FINAL_COLUMNS].to_csv(DATABASE_CSV, index=False)
strict_complete[FINAL_COLUMNS].to_parquet(STRICT_OUTPUT, index=False)
coverage_df.to_csv(COVERAGE_OUTPUT, index=False)

print("Saved:")
print("-", ALL_OUTPUT)
print("-", DATABASE_OUTPUT)
print("-", DATABASE_CSV)
print("-", STRICT_OUTPUT)
print("-", COVERAGE_OUTPUT)
print("\nFinal database rows:", f"{len(retrieval_ready):,}")
print("Strict-complete rows:", f"{len(strict_complete):,}")


Saved:
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed/laptop_catalogue_v2_all.parquet
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed/laptop_product_database_v2.parquet
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed/laptop_product_database_v2.csv
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed/laptop_product_database_v2_strict_complete.parquet
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed/laptop_schema_coverage_v2.csv

Final database rows: 4,938
Strict-complete rows: 4,331


## 16. Reload verification

In [19]:

check = pd.read_parquet(DATABASE_OUTPUT)
assert check["product_id"].is_unique
assert check["price"].notna().all()
assert (check["price"] > 0).all()
assert check["ram_gb"].notna().all()
assert check["storage_gb"].notna().all()
assert check["screen_in"].notna().all()
assert check["gpu"].notna().all()
assert check["description"].notna().all()
print("Reload verification passed.")
print("Rows:", f"{len(check):,}")
display(check.head(20))



# What to send back

Please send these outputs after running:

1. `Laptop category + price candidates`
2. `Excluded by obvious negative title`
3. `Rows kept`
4. Brand distribution
5. RAM values
6. Weight coverage / summary
7. Schema coverage table
8. `Retrieval-ready database` count
9. `Strict-complete subset` count
10. A sample of the final product table

If the retrieval-ready database is comfortably above ~3,000 records and the values look sensible, we should **freeze the data pipeline** and move on to the actual application / FastAPI implementation rather than continuing to optimise preprocessing.
